In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 03 - Qual é o cenário de diversidade de gênero nas carreiras de dados?
# ---------------------------------------------------------------------
caminho_gold_03 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_03_diversidade_genero"

arquivos_gold_03 = [
    str(arquivo) for arquivo in caminho_gold_03.glob("part-*.csv")
]

print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_03)

if not arquivos_gold_03:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_03}"
    )


# Carregar Gold 03
df_genero = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_03)
)


# ---------------------------------------------------------------------
# INSPEÇÃO INICIAL DA GOLD 03 - PARA IDENTIFICAR O QUE EXISTE
# ---------------------------------------------------------------------
"""
A inspeção inicial foi mantida antes de qualquer recorte para validar a estrutura efetivamente disponível na Gold 03. O notebook executado confirmou 346 linhas e sete colunas, com os dados já agregados por edição, gênero, variável e valor, acompanhados de contagem, total de respondentes e percentual na dimensão.
"""
print("\n" + "=" * 100)
print("1. INSPEÇÃO INICIAL DA GOLD 03")
print("=" * 100)

df_genero.show(50, truncate=False)
df_genero.printSchema()

print("Quantidade de linhas:", df_genero.count())
print("Colunas:", df_genero.columns)


# Variáveis existentes
"""
Antes de aprofundar a análise de gênero, são identificadas as dimensões disponíveis na Gold. O output confirmou quatro frentes possíveis de cruzamento: cargo atual, cor/raça/etnia, faixa salarial e nível.
"""
if "variavel" in df_genero.columns:
    print("\nVARIÁVEIS EXISTENTES:")

    (
        df_genero
        .select("variavel")
        .distinct()
        .orderBy("variavel")
        .show(100, truncate=False)
    )


# Edições existentes
"""
A checagem das edições confirma a presença das três pesquisas utilizadas no projeto: 2023-2024, 2024-2025 e 2025-2026. Isso permite que análises posteriores avaliem o cenário atual e também possíveis mudanças ao longo do período.
"""
if "edicao" in df_genero.columns:
    print("\nEDIÇÕES EXISTENTES:")

    (
        df_genero
        .select("edicao")
        .distinct()
        .orderBy("edicao")
        .show(20, truncate=False)
    )


# Valores existentes por variável
"""
Os valores de cada dimensão são inspecionados antes de qualquer agrupamento ou harmonização. Essa etapa é necessária para identificar diferenças de nomenclatura e categorias que podem exigir tratamento específico nas análises seguintes.
"""
print("\n" + "=" * 100)
print("2. VALORES EXISTENTES POR VARIÁVEL")
print("=" * 100)

for variavel in [
    "cargo_atual",
    "cor_raca_etnia",
    "faixa_salarial",
    "nivel"
]:
    print(f"\n{variavel.upper()}:")

    (
        df_genero
        .filter(F.col("variavel") == variavel)
        .select("valor")
        .distinct()
        .orderBy("valor")
        .show(100, truncate=False)
    )


# Gêneros existentes
"""
O output confirma quatro categorias de gênero na base: Feminino, Masculino, Outro e Prefiro não informar. A identificação explícita dessas categorias evita assumir previamente uma estrutura binária para os dados.
"""
print("\nGÊNEROS EXISTENTES:")

(
    df_genero
    .select("genero")
    .distinct()
    .orderBy("genero")
    .show(50, truncate=False)
)


# ---------------------------------------------------------------------
# INSPEÇÃO DETALHADA DE NÍVEL E FAIXA SALARIAL
# ---------------------------------------------------------------------
"""
Nível e faixa salarial são revisados separadamente porque são dimensões que dependem de ordenação para comparações posteriores. A inspeção de nível confirmou quatro categorias padronizadas: Júnior, Pleno, Sênior e Especialista/Staff+.
"""
print("\n" + "=" * 100)
print("3. NÍVEIS EXISTENTES")
print("=" * 100)

(
    df_genero
    .filter(F.col("variavel") == "nivel")
    .select("valor")
    .distinct()
    .orderBy("valor")
    .show(100, truncate=False)
)


"""
A inspeção específica das faixas salariais revelou rótulos que precisam ser observados antes de análises de remuneração. Além das faixas esperadas, aparecem valores como "de R$ 101/mês a R$ 2.000/mês" e "de R$ 25.001/mês a R$ 3000/mês", indicando inconsistências de nomenclatura que não devem ser tratadas automaticamente sem uma regra definida.
"""
print("\n" + "=" * 100)
print("4. FAIXAS SALARIAIS EXISTENTES")
print("=" * 100)

(
    df_genero
    .filter(F.col("variavel") == "faixa_salarial")
    .select("valor")
    .distinct()
    .orderBy("valor")
    .show(100, truncate=False)
)